In [1]:
import json
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor

In [2]:
TRAIN_FEAT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/train_features.csv"
TEST_FEAT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/test_features.csv"
TRAIN_TARGET_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_train.csv"
TEST_TARGET_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_test.csv"
BEST_PARAMS_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/best_params.json"
SUBMISSION_DIR = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/submissions"
SUBMISSION_PATH = f"{SUBMISSION_DIR}/submission_fast.csv"


In [3]:
print("Loading data...")
required_paths = [TRAIN_FEAT_PATH, TEST_FEAT_PATH, TRAIN_TARGET_PATH, TEST_TARGET_PATH]
missing_required = [p for p in required_paths if not os.path.exists(p)]
if missing_required:
    raise FileNotFoundError(f"Missing required files: {missing_required}")

train_features = pd.read_csv(TRAIN_FEAT_PATH)
test_features = pd.read_csv(TEST_FEAT_PATH)
train_target = pd.read_csv(TRAIN_TARGET_PATH)
test_target = pd.read_csv(TEST_TARGET_PATH)
print("Data loaded successfully.")


Loading data...
Data loaded successfully.


In [4]:
# Explicit joins by cust_id so train/test are aligned with CLV files
train_df = train_target.merge(train_features, on="cust_id", how="inner", validate="one_to_one")
test_df = test_target.merge(test_features, on="cust_id", how="left", validate="one_to_one")

# Keep a stable feature set for both train and test
feature_cols = [c for c in train_df.columns if c not in ["cust_id", "revenue_2018_2019"]]
X_train_feat = train_df[feature_cols].copy()
X_test_feat = test_df.reindex(columns=["cust_id"] + feature_cols, fill_value=0).drop(columns=["cust_id"])

y = train_df["revenue_2018_2019"].to_numpy()
y_bin = (y > 0).astype(int)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("X_train shape:", X_train_feat.shape)
print("X_test shape:", X_test_feat.shape)


train_df shape: (116591, 24)
test_df shape: (29148, 23)
X_train shape: (116591, 22)
X_test shape: (29148, 22)


In [5]:
# Load tuned params if available; fallback to defaults
clf_params = {
    "iterations": 400,
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 7,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_state": 42,
    "verbose": False,
}

reg_params = {
    "iterations": 400,
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 7,
    "loss_function": "MAE",
    "random_state": 42,
    "verbose": False,
}

#if os.path.exists(BEST_PARAMS_PATH):
    #with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
        #best_params = json.load(f)
#
    #clf_best = best_params.get("classifier", {}).get("best_params", {})
    #reg_best = best_params.get("regressor", {}).get("best_params", {})
#
    #clf_params.update(clf_best)
    #reg_params.update(reg_best)
    #print(f"Loaded tuned params from: {BEST_PARAMS_PATH}")
#else:
    #print("best_params.json not found. Using default parameters.")
#
#print("Classifier params:", clf_params)
#print("Regressor params:", reg_params)


In [6]:
# Train on all available train data
clf_full = CatBoostClassifier(**clf_params)
clf_full.fit(X_train_feat, y_bin)

train_prob_full = clf_full.predict_proba(X_train_feat)[:, 1]
X_train_reg_full = X_train_feat.copy()
X_train_reg_full["prob_return"] = train_prob_full

reg_full = CatBoostRegressor(**reg_params)

weights = 1 + 3 * train_prob_full   # prueba también 1 + 5 * tr_prob

reg_full.fit(X_train_reg_full, y, sample_weight=weights)

print("Models trained on full training dataset.")


Models trained on full training dataset.


In [11]:
print("Train rows:", X_train_feat.shape[0])
print("Train features:", X_train_feat.shape[1])
print("Test rows:", X_test_feat.shape[0])


Train rows: 116591
Train features: 21
Test rows: 29148


In [7]:
test_prob = clf_full.predict_proba(X_test_feat)[:, 1]
X_test_reg = X_test_feat.copy()
X_test_reg["prob_return"] = test_prob

test = reg_full.predict(X_test_reg)
test_pred = np.clip(test, 0, None)
alpha = 0.2  # el que hayas elegido en validación

test_pred = test_pred  * (alpha * test_prob + (1 - alpha))

submission = test_target[["cust_id"]].copy()
submission["revenue_2018_2019"] = test_pred
os.makedirs(SUBMISSION_DIR, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)
print(submission.head())
print(f"Saved submission: {SUBMISSION_PATH} ({submission.shape})")


            cust_id  revenue_2018_2019
0  2dfoualegmpt6x2h       3.477503e+02
1  d2q2stjpnzld7a4r       3.672086e+00
2  cojscuqlpylhclv2       1.477924e-07
3  vntezlhi2ryvxk6m       1.005128e+02
4  jgy4ytjkdr2b75wf       8.986269e+01
Saved submission: /workspaces/Transactions-predictive-modeling-on-tabular-data-/data/submissions/submission_fast.csv ((29148, 2))
